# Dealer Gamma 04 — hedge flow: does the implied dealer hedge show up in perpetual taker flow?

Pre-registration: `discussions/Dealer_Gamma/001_claude_branch_preregistration.md`. Code: `src/data_engineering/modeling/dealer_gamma.py`.

Per 5-minute bar: predicted dealer hedge $H_t = -100\,G_{t-1}\,r_t/\text{ADV5}$ (positive = dealers should buy). Target: signed perpetual taker flow over the next 15 minutes, scaled by ADV5. OLS $f = a + b\,H_t + c\,r_t + d\,G_{t-1}$, SE clustered by session; 07:50–08:15 UTC excluded.

**H:** b > 0. b reads as the share of the predicted hedge that appears as aggressive perpetual flow. PASS = one-sided p < 0.05 (discovery) / 0.10 (holdout) in both currencies. This is the test that decides whether hedging is established (001 §7).

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import dealer_gamma as dg

ENV = dg.notebook_env('04', PROJECT_ROOT)
OUT, CACHE, HIST = ENV['out'], ENV['cache'], ENV['history']
PERIODS = ['discovery'] + (['holdout'] if ENV['holdout'] else [])
print(dg.VERSION, '| synthetic' if ENV['synthetic'] else '', '| periods', PERIODS)
print('data', ENV['data_root']); print('cache', CACHE); print('out', OUT)

In [ ]:
FITS, BIN = [], []
for c in dg.CURRENCIES:
    d = dg.load_cache(CACHE, c)
    pan = dg.daily_panel(d['grid'], dg.load_dvol(ENV['data_root'], c))
    fp = dg.flow_panel(d['grid'], pan)
    for per in PERIODS:
        for v in dg.VARIANTS:
            FITS.append(dict(dg.flow_fit(fp, per, v, 'f_fwd3'), currency=c))
        FITS.append(dict(dg.flow_fit(fp, per, dg.PRIMARY, 'f_fwd1'), currency=c))
        FITS.append(dict(dg.flow_fit(fp, per, dg.PRIMARY, 'f_now'), currency=c))
        FITS.append(dict(dg.flow_fit(fp, per, dg.PRIMARY, 'f_fwd3', winsor=0.001), currency=c))
        z = fp[fp.period == per].dropna(subset=['h_recon', 'f_fwd3'])
        if len(z) > 1000:
            q = pd.qcut(z.h_recon.rank(method='first'), 20, labels=False)
            BIN.append(z.groupby(q).agg(h_mean=('h_recon', 'mean'), f_mean=('f_fwd3', 'mean'), n=('f_fwd3', 'size'))
                       .reset_index(names='bin').assign(currency=c, period=per))
    print(c, 'bars', len(fp), flush=True)
FITS = pd.DataFrame(FITS); BIN = pd.concat(BIN) if BIN else pd.DataFrame()
FITS.to_csv(OUT / '04_fits.csv', index=False); BIN.to_csv(OUT / '04_binned.csv', index=False)
display(FITS.round(5))

In [ ]:
prim = FITS[(FITS.variant == dg.PRIMARY) & (FITS.target == 'f_fwd3') & (FITS.winsor == 0)].dropna(subset=['p_one_sided'])
dec = dg.decide(prim, ['variant', 'target'])
dg.dump({'registration': dg.registration(), 'decision_primary': dec}, OUT / '04_decision.json')
print(json.dumps(dec, indent=2))

In [ ]:
if len(BIN):
    fig, ax = plt.subplots(1, len(dg.CURRENCIES), figsize=(13, 4.5))
    for j, c in enumerate(dg.CURRENCIES):
        for per in PERIODS:
            z = BIN[(BIN.currency == c) & (BIN.period == per)]
            ax[j].plot(z.h_mean, z.f_mean, 'o-', ms=3, label=per)
        ax[j].axhline(0, color='k', lw=0.6); ax[j].axvline(0, color='k', lw=0.6)
        ax[j].set_xlabel('predicted dealer hedge H (share of ADV5)'); ax[j].set_ylabel('next-15m taker flow (share of ADV5)')
        ax[j].set_title(f'{c}: 20 bins of H'); ax[j].legend()
    plt.tight_layout(); plt.savefig(OUT / '04_binned.png', dpi=110); plt.show()